# GlitchGAN Paper — Figure Refinement

Produces publication-quality versions of all figures for the paper.
Figures are saved to `figures/` in PDF format.

**Sections**
1. Configuration & style
2. Figure 2 — DeepExtractor confusion matrix


## 1. Configuration & style

In [ ]:
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
from IPython.display import Image, display as ipy_display

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Global style — match paper font/size
mpl.rcParams.update({
    "font.family":    "serif",
    "font.size":      10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi":     150,
})

def savefig(fig, name):
    """Save as PDF to figures/ and display inline via BytesIO."""
    path = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(path, bbox_inches="tight")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    ipy_display(Image(buf.read()))
    print(f"Saved: {path}")

print("Config done.")

## 2. Figure 2 — DeepExtractor confusion matrix

Values extracted from `figures/deepextractor_confusion_matrix.png`.
7 true classes (rows) × 16 predicted classes (columns), 100 samples per class.

In [ ]:
TRUE_LABELS = [
    "Blip", "Fast_Scattering", "Koi_Fish", "Low_Frequency_Burst",
    "Scattered_Light", "Tomte", "Whistle",
]

PRED_LABELS = [
    "Air_Compressor", "Blip", "Blip_Low_Frequency", "Extremely_Loud",
    "Fast_Scattering", "Koi_Fish", "Light_Modulation", "Low_Frequency_Burst",
    "Low_Frequency_Lines", "No_Glitch", "Paired_Doves", "Power_Line",
    "Repeating_Blips", "Scattered_Light", "Tomte", "Whistle",
]

# Rows = true label, columns = predicted label
CM_RAW = np.array([
    #AC   Bl   BLF  EL   FS   KF   LM   LFB  LFL  NG   PD   PL   RB   SL   To   Wh
    [ 0, 100,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],  # Blip
    [ 1,   0,   1,   0,  77,   1,   0,   2,   0,   2,   1,   1,   3,   7,   4,   0],  # Fast_Scattering
    [ 0,   2,   0,  14,   0,  79,   1,   0,   0,   0,   0,   0,   4,   0,   0,   0],  # Koi_Fish
    [ 0,   0,   0,   0,   0,   0,   0, 100,   0,   0,   0,   0,   0,   0,   0,   0],  # Low_Frequency_Burst
    [ 0,   0,   0,   0,   3,   0,   0,   2,   5,   7,   0,   0,   0,  83,   0,   0],  # Scattered_Light
    [ 0,   0,   2,   0,   0,   2,   0,   0,   0,   0,   0,   0,   0,   0,  96,   0],  # Tomte
    [ 0,   0,   0,   0,   0,   0,   0,   0,   8,   0,   0,   0,   0,   0,   0,  92],  # Whistle
], dtype=float)

# Sanity check — each row should sum to 100
assert np.all(CM_RAW.sum(axis=1) == 100), "Row sums do not equal 100!"
print("Row sums:", CM_RAW.sum(axis=1))  # should all be 100

In [ ]:
# Drop predicted columns that are all-zero (keep only columns with at least one prediction)
nonzero_cols = CM_RAW.sum(axis=0) > 0
cm_plot = CM_RAW[:, nonzero_cols]
pred_plot = [l for l, keep in zip(PRED_LABELS, nonzero_cols) if keep]

# Normalise to percentage (row-wise) for the heatmap colour, keep raw counts for annotation
cm_pct = cm_plot / cm_plot.sum(axis=1, keepdims=True) * 100

# Annotation: show count; for diagonal add accuracy %
annot = np.empty_like(cm_plot, dtype=object)
for i in range(cm_plot.shape[0]):
    for j in range(cm_plot.shape[1]):
        n = int(cm_plot[i, j])
        if n == 0:
            annot[i, j] = ""
        else:
            annot[i, j] = str(n)

accuracy = np.trace(CM_RAW) / CM_RAW.sum()
print(f"Overall accuracy: {accuracy:.1%}")

fig_w = max(10, len(pred_plot) * 0.9)
fig, ax = plt.subplots(figsize=(fig_w, 5))

sns.heatmap(
    cm_pct,
    annot=annot,
    fmt="",
    cmap="Blues",
    vmin=0, vmax=100,
    cbar_kws={"label": "% of true class"},
    linewidths=0.4,
    linecolor="0.85",
    annot_kws={"size": 8},
    xticklabels=[l.replace("_", "\n") for l in pred_plot],
    yticklabels=[l.replace("_", "\n") for l in TRUE_LABELS],
    ax=ax,
)

ax.set_xlabel("Predicted Label", labelpad=6)
ax.set_ylabel("True Label", labelpad=6)
ax.set_title(f"DeepExtractor — Gravity Spy test set  (accuracy = {accuracy:.1%})")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()

savefig(fig, "deepextractor_confusion_matrix_refined")